In [3]:
import numpy as np
from scipy.special import comb
from scipy.linalg import expm

from electron_integrals import *

from CI import *

from quantum_systems import GeneralOrbitalSystem, ODQD
from configuration_interaction import CISD

Define number of electrons and orbitals

In [4]:
# Number of orbitals (without spin)
num_orbitals = 6
# Number of electrons
num_electrons = 1
#Include spin?
include_spin = False

num_spin_orbitals = (1+int(include_spin))*num_orbitals

Calculate electron integrals

In [5]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

#pot = GaussianWell(w=100, a=1, center=0)
pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a=0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

#Chemistry convention... (Following Hochstuhl)
g = g.transpose(0,2,1,3)

#g = g - g.transpose(0, 1, 3, 2) anti-symmetrisation, code further down not written for this

print('Sanity test, due to symmetry in g this should be zero:')
print(-g[2,1,1,0]+g[2,1,0,1]+g[1,2,1,0]-g[1,2,0,1])

Sanity test, due to symmetry in g this should be zero:
0.0


Solve using address scheme

In [6]:
H = AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E, Psi = np.linalg.eigh(H)
print(E)

[0.49998747 1.49993737 2.49983716 3.49968685 4.49948641 5.49923587]


Solve using Slater-Condon 

In [11]:
H_sc = SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_sc, _ = np.linalg.eigh(H_sc)
print(E_sc)

[ 3.99984969  4.99974948  4.99974948  4.99974948  4.99974948  5.99959917
  5.99959917  5.99959917  5.99959917  5.99964928  5.99969938  5.99969938
  5.99969938  5.99969938  6.99939874  6.99939874  6.99939874  6.99939874
  6.99949896  6.99949896  6.99949896  6.99949896  6.99954906  6.99954906
  6.99954906  6.99954906  6.99959917  6.99959917  6.99959917  6.99959917
  7.99914819  7.99914819  7.99914819  7.99914819  7.99929853  7.99929853
  7.99929853  7.99929853  7.99934863  7.99934863  7.99934863  7.99934863
  7.99934864  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886
  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886
  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886  7.99954907
  8.99904798  8.99904798  8.99904798  8.99904798  8.99909808  8.99909808
  8.99909808  8.99909808  8.99914821  8.99914821  8.99914821  8.99914821
  8.99924842  8.99924842  8.99924842  8.99924842  8.99924842  8.99924842
  8.99924842  8.99924842  8.99924842  8.99924842  8

Comparison with Øyvinds code

In [14]:
odqd = ODQD(num_orbitals, x_max, num_points, alpha=1, a=0.01, potential=HOPotential())
system = GeneralOrbitalSystem(num_electrons, odqd)
cisd = CISD(system, verbose=False).compute_ground_state()
print(cisd.energies)
#print(f'Address scheme difference (rounded to 10 decimals):\n{np.abs(np.round(E-cisd.energies, 10))}')
#print(f'Slater condon difference (rounded to 10 decimals):\n{np.abs(np.round(E-cisd.energies, 10))}')

[ 3.99984969  4.99974948  4.99974948  4.99974948  4.99974948  5.99959917
  5.99959917  5.99959917  5.99959917  5.99964928  5.99969938  5.99969938
  5.99969938  5.99969938  6.99939874  6.99939874  6.99939874  6.99939874
  6.99949896  6.99949896  6.99949896  6.99949896  6.99954906  6.99954906
  6.99954906  6.99954906  6.99959917  6.99959917  6.99959917  6.99959917
  7.99914819  7.99914819  7.99914819  7.99914819  7.99929853  7.99929853
  7.99929853  7.99929853  7.99934863  7.99934863  7.99934863  7.99934863
  7.99934864  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886
  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886
  7.99944886  7.99944886  7.99944886  7.99944886  7.99944886  7.99954907
  8.99904798  8.99904798  8.99904798  8.99904798  8.99909808  8.99909808
  8.99909808  8.99909808  8.99914821  8.99914821  8.99914821  8.99914821
  8.99924842  8.99924842  8.99924842  8.99924842  8.99924842  8.99924842
  8.99924842  8.99924842  8.99924842  8.99924842  8

Is the address scheme better than using Slater-Condon?

In [7]:
a=num_electrons
b=num_spin_orbitals
c=comb(b,a)
print(f'Number of iterations for Slater-Condon: {c*c:.2e}')
print(f'Number of iterations for address scheme: {c*b*b*a*a:.2e}')

Number of iterations for Slater-Condon: 7.84e+02
Number of iterations for address scheme: 7.17e+03


# Imaginary time propagation

In [8]:
def normalize(psi):
    return psi/np.sqrt(psi@psi)

In [9]:
# Choose a random trial state
rng = np.random.default_rng()
psi_trial =  rng.random(Psi[:,0].size)

# Normalize
psi_trial = normalize(psi_trial)

print(f'Overlap with ground state before propagation: {np.abs(psi_trial@Psi[:,0])}')

# Propagation step size
tau = 1
# Propagation operator
U = expm(-H*tau)

for i in range(10000):
    # Propagate
    psi_trial = U@psi_trial
    # Renormalize
    psi_trial = normalize(psi_trial)

# Check that we have converged to the ground state
print(f'Overlap with ground state after propagation: {np.abs(psi_trial@Psi[:,0])}')

Overlap with ground state before propagation: 0.0584488254643209
Overlap with ground state after propagation: 1.0000000000000004
